# SafeTread - ResNet50 Binary Classifier (Colab)
Healthy vs Worn

In [ ]:
# ===== 0) GPU Check =====
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

# ===== 0.5) Mount Google Drive (Colab only) =====
from google.colab import drive
drive.mount('/content/drive')

# ===== 1) Dataset Verification & Counts =====
import os
from collections import defaultdict

DATASET_DIR = '/content/drive/MyDrive/Tyre dataset'
SPLITS = ['train', 'val', 'test']
CLASSES = ['Good', 'Defective']
ALLOWED_EXTS = ('.jpg', '.jpeg', '.png', '.bmp')

def count_images(split_dir):
    counts = defaultdict(int)
    for cls in CLASSES:
        cls_dir = os.path.join(split_dir, cls)
        if not os.path.isdir(cls_dir):
            print(f'⚠ Missing folder: {cls_dir}')
            continue
        files = [f for f in os.listdir(cls_dir) if f.lower().endswith(ALLOWED_EXTS)]
        counts[cls] += len(files)
    return counts

def verify_dataset():
    print('=== Dataset Structure Check ===')
    total_images = 0
    for split in SPLITS:
        split_dir = os.path.join(DATASET_DIR, split)
        if not os.path.isdir(split_dir):
            print(f'❌ Missing split folder: {split_dir}')
            continue
        counts = count_images(split_dir)
        split_total = sum(counts.values())
        total_images += split_total
        print(f'\nSplit: {split}')
        for cls in CLASSES:
            print(f'  {cls}: {counts[cls]} images')
            if counts[cls] == 0:
                print(f'  ⚠ EMPTY folder detected: {split}/{cls}')
        if all(counts[cls] > 0 for cls in CLASSES):
            ratio = max(counts.values()) / min(counts.values())
            if ratio > 1.5:
                print('  ⚠ Class imbalance detected (ratio > 1.5). Consider balancing.')
    print(f'\n✓ Total images across all splits: {total_images}')
    return total_images > 0

has_images = verify_dataset()
if not has_images:
    print('\n⚠ WARNING: No images found! Upload images to your Google Drive.')

# ===== 2) Data Pipeline + Augmentation =====
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_dir = os.path.join(DATASET_DIR, 'train')
val_dir = os.path.join(DATASET_DIR, 'val')
test_dir = os.path.join(DATASET_DIR, 'test')

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    labels='inferred',
    label_mode='binary',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    classes=CLASSES
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    labels='inferred',
    label_mode='binary',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    classes=CLASSES
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    labels='inferred',
    label_mode='binary',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    classes=CLASSES
)

# Apply augmentation to training data only
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
], name='augmentation')

def augment_fn(images, labels):
    images = data_augmentation(images, training=True)
    return images, labels

train_ds = train_ds.map(augment_fn, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.cache().prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.cache().prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.cache().prefetch(tf.data.AUTOTUNE)

# ===== 3) Build ResNet50 Model (Simplified for proper saving) =====
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet import preprocess_input
from tensorflow.keras import layers, models

base_model = ResNet50(
    include_top=False,
    weights='imagenet',
    input_shape=IMG_SIZE + (3,)
)
base_model.trainable = False

model = models.Sequential([
    layers.Input(shape=IMG_SIZE + (3,)),
    layers.Lambda(lambda x: preprocess_input(x)),  # Preprocessing as a layer
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dense(1, activation='sigmoid')
], name='ResNet50_Binary')

model.summary()

# ===== 4) Compile =====
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# ===== 5) Callbacks =====
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint('best_model.h5', monitor='val_loss', save_best_only=True)
]

# ===== 6) Train =====
if has_images:
    EPOCHS = 25
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks
    )
else:
    print('Cannot start training - no images found.')

# ===== 7) Evaluation =====
if has_images:
    import numpy as np
    from sklearn.metrics import confusion_matrix, classification_report
    import matplotlib.pyplot as plt
    
    plt.plot(history.history['accuracy'], label='train_acc')
    plt.plot(history.history['val_accuracy'], label='val_acc')
    plt.title('Training vs Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.show()
    
    y_true = []
    y_pred = []
    for images, labels in test_ds:
        preds = model.predict(images, verbose=0)
        preds = (preds > 0.5).astype(int).flatten()
        y_true.extend(labels.numpy().astype(int))
        y_pred.extend(preds)
    
    cm = confusion_matrix(y_true, y_pred)
    print('Confusion Matrix:')
    print(cm)
    
    print('Classification Report:')
    print(classification_report(y_true, y_pred, target_names=CLASSES))

# ===== 8) Fine-Tuning =====
if has_images:
    base_model.trainable = True
    for layer in base_model.layers[:-20]:
        layer.trainable = False
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    fine_tune_epochs = 8
    history_fine = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=fine_tune_epochs,
        callbacks=callbacks
    )
    
    model.save('best_model_finetuned.h5')
    print('✓ Model saved as best_model_finetuned.h5')
else:
    print('Skipping fine-tuning - no images available')

## 📥 Download Trained Model
Run the cell below to download the trained model to your computer.

In [ ]:
# Download the fine-tuned model
from google.colab import files
files.download('best_model_finetuned.h5')

# Also save to Google Drive as backup
import shutil
shutil.copy('best_model_finetuned.h5', '/content/drive/MyDrive/best_model_finetuned.h5')
print('✓ Model also saved to Google Drive: MyDrive/best_model_finetuned.h5')